In [1]:
import torch
torch.cuda.empty_cache()

In [2]:
from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig

fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
    optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
)

accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

In [3]:
from datasets import load_dataset

train_dataset = load_dataset('json', data_files='input.jsonl', split='train') 

In [4]:
train_dataset

Dataset({
    features: ['context', 'question'],
    num_rows: 13
})

In [5]:
train_dataset[0]['context']

"Deep Learning is a subset of machine learning where artificial neural networks, inspired by the human brain's structure, learn from vast amounts of data. It excels in tasks like image and speech recognition, natural language processing, and decision-making. The training process involves multiple layers (hence 'deep') of interconnected nodes, enabling the model to automatically learn hierarchical features. "

In [6]:
train_dataset[0]['question']

'How does deep learning revolutionize complex pattern recognition tasks?'

In [7]:
test_dataset = load_dataset('json', data_files='input.jsonl', split='train') 

In [8]:
def formatting_fun(example):
    text=f"###Here are the context you need to study and then generate three questions for that: example{'context'} \n\n Generated Question: example{'question'}"
    return text

In [9]:
base_model_id='google/flan-t5-xl'

In [10]:
#load model
import torch
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForSeq2SeqLM.from_pretrained(base_model_id, quantization_config=bnb_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
import warnings
warnings.filterwarnings('ignore')

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
    use_fast=False
)
#tokenizer.pad_token = tokenizer.eos_token

def generate_and_tokenize_prompt(prompt):
    return tokenizer(formatting_fun(prompt))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [13]:
device='cuda' if torch.cuda.is_available() else "cpu"

In [14]:
tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = test_dataset.map(generate_and_tokenize_prompt)

In [15]:
max_length = 30 # This was an appropriate max length for my dataset

def generate_and_tokenize_prompt2(prompt):
    result = tokenizer(
        formatting_fun(prompt),
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

In [16]:
tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt2)
tokenized_val_dataset = test_dataset.map(generate_and_tokenize_prompt2)

In [17]:
#setup Lora
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [18]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [19]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

# Apply the accelerator. You can comment this out to remove the accelerator.
model = accelerator.prepare_model(model)

trainable params: 1093632 || all params: 1743554560 || trainable%: 0.0627242774668319


In [20]:
if torch.cuda.device_count() > 1: # If more than 1 GPU
    model.is_parallelizable = True
    model.model_parallel = True

In [21]:
tokenized_train_dataset.remove_columns(['context','question'])
tokenized_val_dataset.remove_columns(['context','question'])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 13
})

In [22]:
import transformers
from datetime import datetime

project = "finetune"
base_model_name = "googlet5-xl"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name

#tokenizer.pad_token = tokenizer.eos_token

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=1,
        per_device_train_batch_size=5,
        gradient_accumulation_steps=1,
        gradient_checkpointing=True,
        max_steps=500,
        learning_rate=2.5e-5, # Want a small lr for finetuning
        bf16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",        # Directory for storing logs
        save_strategy="steps",       # Save the model checkpoint every logging step
        save_steps=100,                # Save checkpoints every 50 steps
        evaluation_strategy="steps", # Evaluate the model every logging step
        eval_steps=100,               # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to="wandb",           # Comment this out if you don't want to use weights & baises
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"          # Name of the W&B run (optional)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

2024-01-16 08:47:13.466260: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-01-16 08:47:14.210870: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
wandb: Currently logged in as: drnwe. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
100,No log,1.373392
200,No log,1.173568
300,No log,1.013158
400,No log,0.913052
500,0.918500,0.878577


TrainOutput(global_step=500, training_loss=0.9185205078125, metrics={'train_runtime': 1246.1618, 'train_samples_per_second': 4.012, 'train_steps_per_second': 0.401, 'total_flos': 1629254615040000.0, 'train_loss': 0.9185205078125, 'epoch': 250.0})

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig

base_model_id='google/flan-t5-xl'

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    base_model_id,  # Llama 2 7B, same as before
    quantization_config=bnb_config,  # Same quantization config as before
    device_map="auto",
    trust_remote_code=True,
    use_auth_token=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [24]:
from peft import PeftModel
ft_model = PeftModel.from_pretrained(base_model,"googlet5-xl-finetune/checkpoint-500")

In [25]:
context=test_dataset[10]['context']
eval_prompt = f" Generate the questions for me :\n\n### Context: {context}\n\n### Questions: "
eval_prompt

' Generate the questions for me :\n\n### Context: Large Language Models (LLMs) like GPT-5 demonstrate advanced language understanding and generation capabilities. The models excel in tasks requiring contextual comprehension, such as document summarization and sentiment analysis.\n\n### Questions: '

In [26]:
question=test_dataset[10]['question']
question

' What are the key limitations and areas for improvement in Large Language Models to address challenges like bias and misinformation?'

In [27]:
#eval_prompt = "Generate three different questions for me for this context : #  "
model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")
import warnings
warnings.filterwarnings('ignore')

ft_model.eval()
with torch.no_grad():
    output=tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=300)[0], skip_special_tokens=True)
    print(output) 
    print(len(output))

What are the two tasks that the models excel in?
48
